In [ ]:
!pip install transformers==4.38.2 sentence-transformers spacy openai
!python -m spacy download en_core_web_sm

INFO: pip is looking at multiple versions of sentence-transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of sentence-transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 19.2 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.0
    Uninstalling sentence-transformers-5.4.0:
      Successfully uninstalled sentence-transformers-5.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 107.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the pa

In [ ]:
!pip install --force-reinstall \
transformers==4.38.2 \
sentence-transformers==2.6.1 \
huggingface-hub==0.20.3

  Using cached transformers-4.38.2-py3-none-any.whl.metadata (130 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached packaging-26.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.4.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.2 MB/s eta 0:00:00
  Using cached

In [ ]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
import spacy
import random

In [ ]:
nlp = spacy.load("en_core_web_sm")

qg_pipeline = pipeline(
    "text2text-generation",
    model="valhalla/t5-base-qg-hl",
    tokenizer="valhalla/t5-base-qg-hl"
)

embedder = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="your_key",
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
def extract_answers(text):
    doc = nlp(text)
    answers = set()

    for ent in doc.ents:
        answers.add(ent.text)

    return list(answers)

In [ ]:
def highlight_answer(context, answer):
    return context.replace(answer, f"<hl> {answer} <hl>", 1)

In [ ]:
def generate_question(context, answer):
    highlighted = highlight_answer(context, answer)

    input_text = f"generate question: {highlighted}"

    output = qg_pipeline(
        input_text,
        max_length=64,
        num_beams=5,
        early_stopping=True
    )

    question = output[0]['generated_text']

    # clean output
    question = question.replace("<hl>", "")
    question = question.replace("</s>", "")
    question = question.strip()

    return question

In [ ]:
def generate_distractors_llm(question, answer):
    prompt = f"""
    You are generating MCQ options.

    Question: {question}
    Correct Answer: {answer}

    Generate exactly 3 incorrect but plausible answers.
    Keep them similar in type and difficulty.
    Do NOT include the correct answer.

    Return only comma-separated values.
    """

    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )

        text = response.choices[0].message.content.strip()

        # 🔹 Clean parsing
        text = text.replace("\n", ",")
        parts = [p.strip("- ").strip() for p in text.split(",")]

        distractors = []
        for p in parts:
            if p and p.lower() != answer.lower() and p not in distractors:
                distractors.append(p)

        return distractors[:3]

    except Exception as e:
        print("Grok Error:", e)
        return []

In [ ]:
def clean_distractors(distractors):
    cleaned = []

    for d in distractors:
        d = d.strip()

        # remove leading "the"
        if d.lower().startswith("the "):
            d = d[4:]

        # capitalize properly
        d = d[:1].upper() + d[1:]

        if d not in cleaned:
            cleaned.append(d)

    return cleaned

In [ ]:
def is_valid_question(question, answer):
    # reject vague or broken questions
    if len(question) < 10:
        return False

    if answer.lower() in question.lower():
        return False

    # reject weird patterns
    bad_phrases = ["what planet does", "along with"]
    for phrase in bad_phrases:
        if phrase in question.lower():
            return False

    return True

In [ ]:
def generate_mcqs(text):
    answers = extract_answers(text)
    mcqs = []
    seen_questions = set()

    for ans in answers:
        try:
            question = generate_question(text, ans)

            # 🔥 ADD THIS BLOCK HERE
            if not is_valid_question(question, ans):
                continue

            if question in seen_questions:
                continue

            seen_questions.add(question)

            distractors = generate_distractors_llm(question, ans)
            distractors = clean_distractors(distractors)

            # Ensure EXACTLY 4 options
            options = distractors + [ans]
            options = list(set(options))

            while len(options) < 4:
                options.append(random.choice(answers))

            options = options[:4]
            random.shuffle(options)

            mcqs.append({
                "question": question,
                "options": options,
                "answer": ans
            })

        except Exception as e:
            print("Error:", e)
            continue

    return mcqs

In [ ]:
text = """
The Amazon Rainforest, located in South America, is the largest tropical rainforest in the world.
It spans across multiple countries including Brazil, Peru, and Colombia.
The rainforest plays a crucial role in regulating the Earth's climate and is home to millions of species.
"""

mcqs = generate_mcqs(text)

for i, q in enumerate(mcqs, 1):
    print(f"Q{i}: {q['question']}")
    for opt in q['options']:
        print(f" - {opt}")
    print("Answer:", q['answer'])
    print("-" * 50)

Q1: What is the largest rainforest in the world?
 - Congo
 - Brazil
 - Peru
 - Indonesia
Answer: Brazil
--------------------------------------------------
Q2: Where is the Amazon Rainforest located?
 - Australia
 - Central Africa
 - South America
 - Southeast Asia
Answer: South America
--------------------------------------------------
Q3: What is the largest tropical rainforest in the world?
 - Daintree Rainforest
 - The Amazon Rainforest
 - Congo Rainforest
 - Borneo Rainforest
Answer: The Amazon Rainforest
--------------------------------------------------
Q4: How many species live in the Amazon Rainforest?
 - Hundreds of thousands
 - Tens of thousands
 - Billions
 - millions
Answer: millions
--------------------------------------------------
